In [94]:
import tiktoken

In [95]:
text = "Hello! Byte Pair Encoding"
byte_arr = bytearray(text, "utf-8")
print(byte_arr)

bytearray(b'Hello! Byte Pair Encoding')


In [96]:
ids = list(byte_arr)
print(ids)

[72, 101, 108, 108, 111, 33, 32, 66, 121, 116, 101, 32, 80, 97, 105, 114, 32, 69, 110, 99, 111, 100, 105, 110, 103]


In [97]:
print("Number of characters", len(text))
print("Number of Bytes", len(ids))

Number of characters 25
Number of Bytes 25


Using Tiktoken

In [98]:
enc = tiktoken.encoding_for_model("gpt-4o")
tokens = enc.encode(text)
print(tokens)

[13225, 0, 20445, 41250, 70820]


Load Datatext

In [99]:
# I would be using the shakespeare text

def load_shakespeare():
    file_path = "/home/bukunmi/ml-journey/datasets/tinyshakespeare.txt"

    with open(file_path, "r", encoding="utf-8") as f:
        raw = f.read()
    return raw

In [100]:
text = load_shakespeare()
len(text)

1115394

In [101]:
from collections import Counter

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

class BPETokenizer:
    def __init__(self, vocab_size=50000):
        if vocab_size < 256:
            raise ValueError("byte-level BPE needs vocab_size >= 256")
        self.vocab_size = vocab_size
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        self.inverse_vocab = {value: idx for idx, value in self.vocab.items()}
        self.merges = {}

    def _reset_vocab(self):
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        self.inverse_vocab = {value: idx for idx, value in self.vocab.items()}
        self.merges = {}

    def _get_pair_frequencies(self, ids):
        return Counter(zip(ids, ids[1:]))
    
    def _merge_pair(self, ids, pair, new_id):
        merged = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                merged.append(new_id)
                i += 2
            else:
                merged.append(ids[i])
                i += 1
        return merged
    
    def fit(self, text):
        self._reset_vocab()
        ids = list(text.encode("utf-8"))
        if len(ids) < 2:
            return self

        total_merges = self.vocab_size - 256
        steps = range(total_merges)
        if tqdm is not None:
            steps = tqdm(steps, desc="Training BPE", unit="merge")

        for _ in steps:
            pair_counts = self._get_pair_frequencies(ids)
            if not pair_counts:
                break

            best_pair = max(pair_counts.items(), key=lambda item: (item[1], item[0]))[0]
            new_id = 256 + len(self.merges)
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.inverse_vocab[self.vocab[new_id]] = new_id
            ids = self._merge_pair(ids, best_pair, new_id)

        return self
    
    def tokenize(self, text):
        ids = list(text.encode("utf-8"))
        for pair, new_id in self.merges.items():
            if len(ids) < 2:
                break
            ids = self._merge_pair(ids, pair, new_id)
        return ids
    
    def encode(self, text):
        return self.tokenize(text)
    
    def decode(self, tokens, errors="strict"):
        try:
            text_bytes = b"".join(self.vocab[token] for token in tokens)
        except KeyError as exc:
            raise ValueError(f"unknown token id: {exc.args[0]}") from None
        return text_bytes.decode("utf-8", errors=errors)

In [102]:
tokenizer = BPETokenizer(10000)

In [ ]:
tokenizer.fit(text)

Training BPE:   0%|          | 0/9744 [00:00<?, ?merge/s]

In [ ]:
test_text = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

Hello! It is Bukunmi.
The quick brown fox jumps over the lazy dog.
Machine learning models tokenize text into smaller pieces.
Spaces,    tabs,	and newlines
should survive exactly.

Unicode test: café, naïve, résumé, Ж, 🚀, 😀.
Rare name test: Bukunmi Akinyemi.
Repeated pattern test: token token token tokenization tokenize tokenizer.
"""

In [ ]:
print("Test text length: ", len(test_text))

Test text length:  416


In [ ]:
tokens = tokenizer.encode(test_text)
print("My BPE: ", len(tokens))

My BPE:  224


In [ ]:
tik_tokens = enc.encode(test_text)
print("Tiktoken: ", len(tik_tokens))

Tiktoken:  101


In [ ]:
tokenizer.decode(tokens)

'\nFirst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nHello! It is Bukunmi.\nThe quick brown fox jumps over the lazy dog.\nMachine learning models tokenize text into smaller pieces.\nSpaces,    tabs,\tand newlines\nshould survive exactly.\n\nUnicode test: café, naïve, résumé, Ж, 🚀, 😀.\nRare name test: Bukunmi Akinyemi.\nRepeated pattern test: token token token tokenization tokenize tokenizer.\n'

In [ ]:
enc.decode(tik_tokens)

'\nFirst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nHello! It is Bukunmi.\nThe quick brown fox jumps over the lazy dog.\nMachine learning models tokenize text into smaller pieces.\nSpaces,    tabs,\tand newlines\nshould survive exactly.\n\nUnicode test: café, naïve, résumé, Ж, 🚀, 😀.\nRare name test: Bukunmi Akinyemi.\nRepeated pattern test: token token token tokenization tokenize tokenizer.\n'